# 3-3절 연습 문제 풀이

이 노트북은 3-3절 연습 문제의 풀이 예시다. 정답이 하나뿐인 문제가 아니므로 다른 구현도 얼마든지 가능하다.

- 본문 예제 코드는 `notebooks/ch03/` 아래 예제 노트북을 참고한다.
- 위에서부터 차례대로 실행한다.

In [ ]:
# 환경 설정 - 공통 라이브러리, 시드 고정, 장치 객체
import sys
sys.path.append('../../')

import random

import numpy as np
import torch
import torch.nn as nn

from code_reference import common
# viz.configure()에서 save_grayscale=True로 지정하면 노트북에 표시되는 시각화 이미지를 파일로 저장함
from code_reference import visualize as viz

viz.configure(save_grayscale=False)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = common.get_device()

# 연습 문제에서 공통으로 사용하는 CSV 로더
# 본문 데이터(ch3_spiral_data.csv)는 열 이름이 x1, x2 이고 정답이 숫자,
# 연습 문제 데이터(ch3_exercise_*.csv)는 열 이름이 x, y 이고 정답이 문자열이다.
import csv

def load_csv(path):
    with open(path, encoding='utf-8') as f:
        rows = list(csv.DictReader(f))
    cols = rows[0].keys()
    cx, cy = ('x1', 'x2') if 'x1' in cols else ('x', 'y')
    xs = [[float(r[cx]), float(r[cy])] for r in rows]
    labels = [r['label'] for r in rows]
    classes = sorted(set(labels), key=lambda v: (float(v) if v.replace('.','',1).isdigit() else v))
    idx = {name: i for i, name in enumerate(classes)}
    X = torch.tensor(xs)
    Y = torch.tensor([idx[l] for l in labels])      # 교차 엔트로피용 정수 정답
    return X, Y, classes

## 연습 3-10

[코드 3-24]에서 모델의 정확도를 계산할 때 정답을 맞힌 샘플의 수를 집계한 후 전체 샘플의 수로 나눠 백분율을 구하는 두 줄의 코드는, (classes == Y).float().mean().item() * 100 한 줄로 줄일 수 있다. 이 한 줄의 코드가 정확도 백분율을 계산하는 과정을 풀어서 설명해 보자.

In [ ]:
classes = torch.tensor([0, 1, 2, 1, 0])      # 예측 클래스
Y = torch.tensor([0, 1, 1, 1, 0])            # 정답

eq = (classes == Y)                 # 1) 맞으면 True, 틀리면 False
print(f'1) 비교 결과   : {eq.tolist()}')
f = eq.float()                      # 2) True -> 1.0, False -> 0.0
print(f'2) float() 변환: {f.tolist()}')
m = f.mean()                        # 3) 평균 = 맞힌 수 / 전체 수
print(f'3) mean()      : {m.item():.2f}  (= 4 / 5)')
print(f'4) item() * 100: {m.item() * 100:.2f}%')

불리언 텐서를 실수형으로 바꾸면 True는 1.0, False는 0.0이 된다. 이 텐서의 **평균**이 곧 (맞힌 샘플 수 ÷ 전체 샘플 수)이므로, 100을 곱하면 바로 백분율 정확도가 된다. 합계를 구해 전체 수로 나누는 두 줄을 한 줄로 줄인 것이다.

## 연습 3-11

교차 엔트로피 손실 함수를 사용해 학습하는 모델은 출력층에 활성화 계층을 덧붙이지 않는다. 그렇다면 분류 모델이 아닌 1장의 회귀 분석 모델에서 활성화 함수를 사용하지 않았던 이유가 무엇인지 생각해 보자.

### 풀이

**분류 모델에서 출력층에 활성화 계층을 붙이지 않는 이유**
`nn.CrossEntropyLoss`가 내부에서 소프트맥스를 함께 계산하기 때문이다. 모델에 소프트맥스를 또 붙이면 두 번 적용되어 확률 분포가 뭉개지고 수치적으로도 불안정해진다.

**회귀 분석 모델에서 활성화 함수를 쓰지 않은 이유는 다르다**
1장 회귀 모델의 출력은 낙하거리처럼 **범위 제한이 없는 실숫값**이다. 시그모이드(0~1)나 소프트맥스(합이 1)를 붙이면 출력 범위가 인위적으로 제한되어 원하는 값을 낼 수 없다.

| 구분 | 출력층에 활성화 함수를 쓰지 않는 이유 |
|---|---|
| 분류 모델 | 손실 함수가 소프트맥스를 **이미 포함**하고 있어서 |
| 회귀 모델 | 출력값의 **범위를 제한하면 안 되므로** |

즉 출력층의 활성화 함수는 "풀려는 문제의 출력이 어떤 값이어야 하는가"에 따라 정해진다.

## 연습 3-12

[코드 3-26]과 [코드 3-29]에서 만드는 모델 객체를 다음과 같은 방법으로도 만들어 보자.

nn.Sequential을 사용하지 않고 모델 클래스로 정의해서 생성

선형 계층과 활성화 계층의 조합을 만들어 사용

In [ ]:
X, Y, classes = load_csv('../../data/ch3_spiral_data.csv')
N_CLASSES = len(classes)
print(f'샘플 {len(X)}개, 클래스 {N_CLASSES}개 {classes}')

# 방법 1) 모델 클래스로 정의
class SpiralClassifier(nn.Module):
    def __init__(self, hidden_dim=16, n_classes=N_CLASSES):
        super().__init__()
        self.fc1 = nn.Linear(2, hidden_dim)
        self.act = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, n_classes)

    def forward(self, x):
        return self.fc2(self.act(self.fc1(x)))

# 방법 2) 선형 계층과 활성화 계층의 조합을 함수로 만들어 재사용
def linear_block(fan_in, fan_out):
    return nn.Sequential(nn.Linear(fan_in, fan_out), nn.ReLU())

for name, make in [('모델 클래스', lambda: SpiralClassifier()),
                   ('블록 조합', lambda: nn.Sequential(linear_block(2, 16),
                                                    nn.Linear(16, N_CLASSES)))]:
    torch.manual_seed(SEED)
    model = make()
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.05)
    for _ in range(1000):
        loss = criterion(model(X), Y)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
    acc = ((model(X).argmax(dim=1) == Y).float().mean() * 100).item()
    print(f'{name:10s} 정확도 {acc:5.2f}%, '
          f'파라미터 {sum(p.numel() for p in model.parameters())}개')

세 방식(`nn.Sequential` 직접 나열, 모델 클래스, 블록 조합)은 **구조와 파라미터 수가 같다**. 모델 클래스는 `forward()`에서 흐름을 자유롭게 제어할 수 있고, 블록 조합은 반복되는 구조를 재사용하기 좋다. 5장 이후의 합성곱 블록이 블록 조합 방식의 대표적인 예다.

## 연습 3-13

깃허브 저장소의 data 디렉터리에 있는 ch3_exercise_3.csv 파일과 ch3_exercise_4.csv 파일은 각각 [연습 문제 3-8]의 ch3_exercise_1.csv, ch3_exercise_2.csv 파일과 성격은 비슷하지만, 더 많은 클래스로 분류되는 다중 클래스 데이터다. 두 파일을 그래프로 나타내면 [그림 3-16]과 같다.

그림 3-16 ch3_exercise_3.csv와 ch3_exercise_4.csv 파일에 저장된 데이터의 분포

ch3_exercise_3.csv 파일의 데이터를 분류할 수 있는 모델과 ch3_exercise_4.csv 파일의 데이터를 분류할 수 있는 모델을 만들어 보자.

In [ ]:
def train_multiclass(path, hidden=32, epochs=2000, lr=0.05):
    X, Y, classes = load_csv(path)
    torch.manual_seed(SEED)
    model = nn.Sequential(
        nn.Linear(2, hidden), nn.ReLU(),
        nn.Linear(hidden, hidden), nn.ReLU(),
        nn.Linear(hidden, len(classes)),        # 출력 크기 = 클래스 수
    )
    criterion = nn.CrossEntropyLoss()           # 소프트맥스 포함
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    for _ in range(epochs):
        loss = criterion(model(X), Y)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
    acc = ((model(X).argmax(dim=1) == Y).float().mean() * 100).item()
    print(f'{path.split("/")[-1]}: 클래스 {len(classes)}개 {classes} -> 정확도 {acc:.2f}%')

for name in ['ch3_exercise_3.csv', 'ch3_exercise_4.csv']:
    train_multiclass(f'../../data/{name}')

이진 분류에서 다중 클래스로 바꿀 때 달라지는 부분은 세 가지다.

1. **출력 크기**를 클래스 수로 (1 → 3 또는 4)
2. **손실 함수**를 `nn.BCELoss` → `nn.CrossEntropyLoss`로
3. **정답 텐서**를 (N, 1) 실수형 → (N,) 정수형(클래스 인덱스)으로

출력층에 활성화 계층을 붙이지 않는 점도 함께 기억해 두자.

## 연습 3-14

[도전 문제] ReLU 활성화 함수와 교차 엔트로피 손실 함수를 직접 구현하고, 이를 사용해 [코드 3-29]와 같은 회오리 모양 데이터 분류 모델을 만들어 보자.

3장 학습 노트

In [ ]:
# ReLU와 교차 엔트로피 손실을 직접 구현한다.
def my_relu(x):
    return torch.maximum(x, torch.zeros_like(x))

def my_cross_entropy(logits, target):
    # 1) 수치 안정성을 위해 최댓값을 뺀 뒤 로그 소프트맥스를 계산
    shifted = logits - logits.max(dim=1, keepdim=True).values
    log_probs = shifted - shifted.exp().sum(dim=1, keepdim=True).log()
    # 2) 정답 클래스의 로그 확률만 모아 부호를 바꾸고 평균
    picked = log_probs[torch.arange(len(target)), target]
    return -picked.mean()

logits = torch.randn(5, 3)
target = torch.tensor([0, 2, 1, 1, 0])
print(f'직접 구현: {my_cross_entropy(logits, target).item():.6f}')
print(f'파이토치 : {nn.functional.cross_entropy(logits, target).item():.6f}')
print(f'ReLU 일치: {torch.equal(my_relu(logits), torch.relu(logits))}')

In [ ]:
# 직접 구현한 함수로 회오리 데이터 분류 모델을 학습한다.
X, Y, classes = load_csv('../../data/ch3_spiral_data.csv')
torch.manual_seed(SEED)

W1 = (torch.randn(2, 32) * 0.5).requires_grad_(True)
b1 = torch.zeros(32, requires_grad=True)
W2 = (torch.randn(32, len(classes)) * 0.5).requires_grad_(True)
b2 = torch.zeros(len(classes), requires_grad=True)

optimizer = torch.optim.Adam([W1, b1, W2, b2], lr=0.05)
for epoch in range(2000):
    hidden = my_relu(X @ W1 + b1)
    logits = hidden @ W2 + b2
    loss = my_cross_entropy(logits, Y)
    optimizer.zero_grad(); loss.backward(); optimizer.step()

acc = ((logits.argmax(dim=1) == Y).float().mean() * 100).item()
print(f'직접 구현한 모델 정확도: {acc:.2f}% (최종 손실 {loss.item():.4f})')

두 함수 모두 파이토치 구현과 같은 값을 낸다. 교차 엔트로피는 `-log(정답 클래스의 확률)`의 평균인데, 지수 계산에서 값이 넘치지 않도록 **최댓값을 빼는 보정**을 넣는 것이 실무 구현의 핵심이다. 파이토치의 `cross_entropy`도 내부에서 같은 처리를 한다.